In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf

sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic import NMF_logistic


gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  # Restrict TensorFlow to only allocate 1GB of memory on the first GPU
  try:
    tf.config.experimental.set_virtual_device_configuration(
        gpus[0],
        [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=8192)])
    logical_gpus = tf.config.experimental.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Virtual devices must be set before GPUs have been initialized
    print(e)

sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_2020_4mice_nan.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
#group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
#behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)

indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos


In [ ]:
sys.path.append('/home/austin/Basic')
from utils_np import safe_softplus

In [ ]:
coherence.shape

In [ ]:
granger.shape

In [ ]:
myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
#group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
#behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)

indx_pos = (behavior==1)&(condition==4)
indx_neg1 = (behavior==2)&(condition==4)
indx_neg2 = (behavior==2)&(condition==6)
indx_neg3 = (behavior==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos


In [ ]:
granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6

X = np.hstack((power,coherence,granger))


In [ ]:
y = np.zeros(len(behavior))
y[indx_pos] = 1

X = X[indx_tot]
time = time[indx_tot]
y = y[indx_tot]
mouse = mouse[indx_tot]


In [ ]:
md = pickle.load(open('Unbalanced_Elastic_12_enc_1.0.p','rb'))

In [ ]:
md.keys()

In [ ]:
A_enc = md['A_enc']
B_enc = md['B_enc']
A_enc.shape

In [ ]:
S_est = safe_softplus(np.dot(X,A_enc) + B_enc)


In [ ]:
myDict2 = {'mouse':mouse,'Scores_supervised':S_est[:,0],
           'Scores_unsupervised':S_est[:,1:],'time':time,'condition':y}
from scipy.io import savemat
savemat('Scores_4mice_nan.mat',myDict2)